# ChatGPT Second-Round Sense Assignment Pipeline

This notebook implements a robust, efficient, and maintainable pipeline for second-round sense assignment using RAG (retrieval-augmented generation) and LLMs. Key features:
- Unified single-pass logic for both MWEs and single tokens.
- Uses OpenAI embeddings (via LangChain) with metadata filtering and robust caching (aligned by S_ID).
- Presents top-N candidate senses to ChatGPT for selection; generates new senses if needed.
- Maintains an in-memory DataFrame for new senses, using consistent field names from config.py.
- Avoids duplicate new senses and logs all new senses to a JSONL file.
- All field names are imported from config.py for consistency.
- Embedding and LLM operations use LangChain wrappers.
- Designed for repeatable, efficient, and traceable sense assignment.

# Imports and Configuration

# Second Round Sense Assignment with ChatGPT and RAG

This notebook performs a second round of sense assignment for sentences where the LLM failed to find a suitable sense (i.e., `SENSE_ORIGIN` is 'FIRST'). It uses a multi-pass approach:

1. **RAG (Retrieval-Augmented Generation):**
   - Filter sense repository by relevant metadata (lemma, POS, etc.).
   - Use OpenAI embeddings for semantic similarity search.
   - If a close match is found, link the lemma to the sense and log the action.
2. **LLM Generation:**
   - For cases where no suitable sense is found, prompt ChatGPT to generate a new sense definition.
   - Check for near-duplicates before adding.
   - Log all new senses in a separate repository for review.

All actions (linking, new sense creation) are logged for traceability.

In [1]:
# 1. Import Required Libraries and Field Names
import pandas as pd
import numpy as np
import json
import os
import pickle
from pathlib import Path
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from config import (
    OPENAI_API_KEY, SENSE_REPO, OUTPUT_DIR,
    S_ID, S_LEMMA, S_DEFINITION, S_TYPE, S_POS, SENSE_ID_FIELD, SENSE_ORIGIN,
    L_MWE_TYPE, L_MWE_LEMMA, L_POS, L_LEMMA,
    SENSE_AINOTES_FIELD, SENSE_LIST_FIELD, SENSE_COUNT_FIELD
)
from process_senses import token_is_not_in_mwe

In [ ]:
# 2. Load Sense Repository and Filtered Sentences
from data_loader import load_sense_repo
senses_df = load_sense_repo()
# Keep only columns actually used in the pipeline
senses_df = senses_df[[S_ID, S_DEFINITION, S_TYPE, S_POS]].copy()
# Remove duplicates that were only linked to different lemmas
senses_df = senses_df.drop_duplicates(subset=[S_DEFINITION, S_TYPE, S_POS]).reset_index(drop=True)

filtered_path = OUTPUT_DIR / "filtered_first_origin_incept_ChatGPT-4-1.tsv"
from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser
parser = WebAnnoLEXISParser(filtered_path)
filtered_sentences = parser.parse()

In [3]:
# 3. Embedding Caching (robust, S_ID-aligned)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=OPENAI_API_KEY)

# Reduce senses_df to only the required columns and deduplicate
senses_df = senses_df[[S_ID, S_DEFINITION, S_TYPE, S_POS]].drop_duplicates(subset=[S_DEFINITION, S_TYPE, S_POS])

def embed_text(text):
    return np.array(embeddings.embed_query(text))

embeddings_path = OUTPUT_DIR / "sense_embeddings.pkl"
if os.path.exists(embeddings_path):
    with open(embeddings_path, "rb") as f:
        id_to_emb = pickle.load(f)
    senses_df['embedding'] = senses_df[S_ID].map(id_to_emb)
else:
    id_to_emb = {row[S_ID]: embed_text(row[S_DEFINITION]) for _, row in senses_df.iterrows()}
    with open(embeddings_path, "wb") as f:
        pickle.dump(id_to_emb, f)
    senses_df['embedding'] = senses_df[S_ID].map(id_to_emb)

In [4]:
print(f"senses_df columns: {list(senses_df.columns)}")

senses_df columns: ['senseID', 'definition', 'Type', 'pos', 'embedding']


In [5]:
# 4. Utility Functions: Cosine Similarity, LLM Invocation, Logging, AI Sense ID (updated for two chains)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def generate_ai_sense_id(existing_ids, prefix="AI_ChatGPT-"):
    i = 1
    while True:
        candidate = f"{prefix}{i:04d}"
        if candidate not in existing_ids:
            return candidate
        i += 1

def log_new_senses(new_senses_df, log_path):
    """Append new senses to a JSONL log file."""
    with open(log_path, 'a', encoding='utf-8') as f:
        for _, row in new_senses_df.iterrows():
            f.write(json.dumps(row.to_dict(), ensure_ascii=False) + '\n')

def build_senses_block(senses):
    lines = [f"{i+1}. ID: {row[S_ID]} — {row[S_DEFINITION]}" for i, row in senses.iterrows()]
    return "\n".join(lines)

def llm_select_sense(context, lemma, senses_block, selection_chain):
    response = selection_chain.invoke({
        "context": context,
        "lemma": lemma,
        "senses_block": senses_block
    })
    return json.loads(response)

def llm_generate_new_sense(context, lemma, pos, generation_chain):
    response = generation_chain.invoke({
        "context": context,
        "lemma": lemma,
        "pos": pos or ""
    })
    return json.loads(response)

In [6]:
# 5. Initialize LLM and Prompts (adapted to match first round style)
llm = ChatOpenAI(temperature=0, openai_api_key=OPENAI_API_KEY, name="gpt-4.1-nano-2025-04-14")

# Selection prompt: strictly select from provided senses (as in first round)
selection_system_message = """
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:
{{
  "sense_id": "<jedan od ponuđenih ID-jeva ili 'NEW_SENSE'>",
  "explanation": "<kratko i jasno obrazloženje u jednoj ili dve rečenice zašto je to značenje primenjivo.\nAko se koristi 'NEW_SENSE', objasnite zašto nijedno ponuđeno značenje ne odgovara.>"
}}

Ne dodajete nikakav dodatni tekst van JSON strukture.
Koristite 'NEW_SENSE' samo ako nijedno značenje nije čak ni približno tačno u kontekstu.
"""
selection_user_prompt = """
Kontekst rečenice (ciljna reč je označena HTML tagom <b>...</b>):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}
"""
selection_prompt = ChatPromptTemplate.from_messages([
    ("system", selection_system_message),
    ("user", selection_user_prompt)
])
selection_parser = StrOutputParser()
selection_chain = selection_prompt | llm | selection_parser

# Generation prompt: create a new sense definition (as in first round, but for new sense only)
generation_system_message = """
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i gramatičkih informacija,
kreirajte novo značenje za datu reč jer nijedno postojeće nije odgovarajuće.
Odgovor mora biti u JSON formatu:
{{
  "definition": "<definicija>",
  "explanation": "<kratko obrazloženje>"
}}
"""
generation_user_prompt = """
Kontekst rečenice: "{sentence}"
Ciljna reč: "{word}"
Gramatika: {pos}
"""
generation_prompt = ChatPromptTemplate.from_messages([
    ("system", generation_system_message),
    ("user", generation_user_prompt)
])
generation_parser = StrOutputParser()
generation_chain = generation_prompt | llm | generation_parser

In [7]:
# Utility: Build marked sentence for prompt (as in first round)
def build_marked_sentence(context, target, mark_tag="b"):
    # Mark the target word in the context with <b>...</b>
    # This is a simple version; for MWEs, you may want to mark all tokens
    return context.replace(target, f"<{mark_tag}>{target}</{mark_tag}>")

In [8]:
# 6. In-Memory New Senses Repository (standardized columns)
new_senses_columns = [S_ID, S_LEMMA, S_DEFINITION, S_TYPE, S_POS, 'source']
new_senses_df = pd.DataFrame(columns=new_senses_columns)

In [9]:
# Ensure all field names are config-driven in senses_df and new_senses_df setup
# (This cell typically prepares senses_df and new_senses_df)



# If new_senses_df exists, ensure its columns are config-driven as well
if 'new_senses_df' in locals():
    new_senses_df = new_senses_df[[S_ID, S_LEMMA, S_DEFINITION, S_TYPE, S_POS]].drop_duplicates(subset=[S_DEFINITION, S_TYPE, S_POS])

print(f"senses_df columns: {list(senses_df.columns)}")
if 'new_senses_df' in locals():
    print(f"new_senses_df columns: {list(new_senses_df.columns)}")

len_filtered = len(filtered_sentences)
print(f"Processing {len_filtered} filtered sentences...")

senses_df columns: ['senseID', 'definition', 'Type', 'pos', 'embedding']
new_senses_df columns: ['senseID', 'lemma', 'definition', 'Type', 'pos']
Processing 279 filtered sentences...


In [10]:
# 7. Unified Single-Pass Sense Assignment Pipeline (adapted for marked sentence and prompt style)
TOP_N = 7
new_sense_log_path = OUTPUT_DIR / "ai_generated_senses.jsonl"
n = 2000  # Unique index for multi-token entities

for i, sentence in enumerate(filtered_sentences):
    print(f"\nProcessing sentence {i+1}/{len(filtered_sentences)}: {getattr(sentence, 'text', None)}")
    # --- Process MWEs ---
    print(f"  Number of MWEs: {len(getattr(sentence, 'mwes', []))}")
    for mwe in sentence.mwes:
        mwe_tokens = getattr(mwe, 'tokens', []) if hasattr(mwe, 'tokens') else []
        mwe_first = any(token.layers.get(SENSE_ORIGIN, None) and "FIRST" in token.layers[SENSE_ORIGIN] for token in mwe_tokens)
        if not mwe_first:
            continue
        print(f"    [MWE] FIRST detected: {getattr(mwe, L_MWE_LEMMA, None)} | tokens: {len(mwe_tokens)}")
        lemma = getattr(mwe, L_MWE_LEMMA, None)
        mwe_type = getattr(mwe, L_MWE_TYPE, None)
        context = getattr(sentence, 'text', None)
        marked_sentence = build_marked_sentence(context, lemma)
        print(f"    MWE lemma: {lemma}, type: {mwe_type}, tokens: {len(mwe_tokens)}")
        # Step 1: Check new_senses_df for this lemma/type
        candidates = new_senses_df[(new_senses_df[S_LEMMA] == lemma) & (new_senses_df[S_TYPE] == mwe_type)]
        print(f"      New senses candidates: {len(candidates)}")
        senses_block = build_senses_block(candidates) if not candidates.empty else ""
        senses_candidates = ";".join(candidates[S_ID].tolist())
        senses_count = str(len(candidates))
        if not candidates.empty:
            result = selection_chain.invoke({
                "sentence": marked_sentence,
                "word": lemma,
                "senses_block": senses_block
            })
            result = json.loads(result)
            explanation = result.get("explanation", "")
            if result.get("sense_id") != "NEW_SENSE":
                match = candidates[candidates[S_ID] == result.get("sense_id")]
                if not match.empty:
                    for token in mwe_tokens:
                        token.layers[SENSE_ID_FIELD] = f"{match.iloc[0][S_ID]}[{n}]"
                        token.layers[SENSE_ORIGIN] = 'AI-NEW-REPO'
                        token.layers[SENSE_AINOTES_FIELD] = f"{explanation}[{n}]"
                        token.layers[SENSE_LIST_FIELD] = f"{senses_candidates}[{n}]"
                        token.layers[SENSE_COUNT_FIELD] = f"{senses_count}[{n}]"
                    n += 1
                    continue
        # Step 2: RAG over main repo
        # Always include 'embedding' column in filtered DataFrame
        filter_cols = [S_ID, S_DEFINITION, S_TYPE, S_POS, 'embedding']
        filtered = senses_df[filter_cols]
        if mwe_type and S_TYPE in senses_df.columns:
            filtered = filtered[filtered[S_TYPE] == mwe_type]
        print(f"      RAG candidate senses: {len(filtered)}")
        if len(filtered) == 0:
            print("      WARNING: No RAG candidates found for this MWE.")
        # --- Embedding lookup and update logic ---
        context_emb = id_to_emb.get(lemma) if lemma in id_to_emb else None
        if context_emb is None:
            context_emb = embed_text(context)
            # Add the new embedding to id_to_emb for future use
            id_to_emb[lemma] = context_emb
        filtered = filtered.copy()  # Avoid SettingWithCopyWarning
        filtered['sim'] = filtered['embedding'].apply(lambda x: cosine_similarity(context_emb, x))
        top_senses = filtered.sort_values('sim', ascending=False).head(TOP_N)
        # Print the top five most semantically relevant candidates after similarity calculation
        if len(top_senses) > 0:
            print("      Top RAG candidates (by similarity):")
            print(top_senses[[S_ID, S_DEFINITION, S_TYPE, S_POS, 'sim']].head())
        senses_block = build_senses_block(top_senses)
        senses_candidates = ";".join(top_senses[S_ID].tolist())
        senses_count = str(len(top_senses))
        result = selection_chain.invoke({
            "sentence": marked_sentence,
            "word": lemma,
            "senses_block": senses_block
        })
        result = json.loads(result)
        explanation = result.get("explanation", "")
        if result.get("sense_id") != "NEW_SENSE":
            match = top_senses[top_senses[S_ID] == result.get("sense_id")]
            if not match.empty:
                best = match.iloc[0]
                for token in mwe_tokens:
                    token.layers[SENSE_ID_FIELD] = f"{best[S_ID]}[{n}]"
                    token.layers[SENSE_ORIGIN] = 'RAG-OLD-REPO'
                    token.layers[SENSE_AINOTES_FIELD] = f"{explanation}[{n}]"
                    token.layers[SENSE_LIST_FIELD] = f"{senses_candidates}[{n}]"
                    token.layers[SENSE_COUNT_FIELD] = f"{senses_count}[{n}]"
                n += 1
                continue
        # Step 3: Generate new sense
        result = generation_chain.invoke({
            "sentence": marked_sentence,
            "word": lemma,
            "pos": mwe_type or ""
        })
        result = json.loads(result)
        explanation = result.get("explanation", "")
        existing_ids = set(senses_df[S_ID]).union(set(new_senses_df[S_ID]))
        new_id = generate_ai_sense_id(existing_ids)
        new_entry = {
            S_ID: new_id,
            S_LEMMA: lemma,
            S_DEFINITION: result['definition'],
            S_TYPE: mwe_type,
            S_POS: None,
            'source': 'AI-GENERATED'
        }
        new_senses_df = pd.concat([new_senses_df, pd.DataFrame([new_entry])], ignore_index=True)
        log_new_senses(pd.DataFrame([new_entry]), new_sense_log_path)
        for token in mwe_tokens:
            token.layers[SENSE_ID_FIELD] = f"{new_id}[{n}]"
            token.layers[SENSE_ORIGIN] = 'AI-GENERATED'
            token.layers[SENSE_AINOTES_FIELD] = f"{explanation}[{n}]"
            token.layers[SENSE_LIST_FIELD] = f"{new_id}[{n}]"
            token.layers[SENSE_COUNT_FIELD] = "1"
        n += 1
    # --- Process single tokens not in MWEs ---
    print(f"  Number of tokens: {len(sentence.tokens)}")
    for token in sentence.tokens:
        if not token_is_not_in_mwe(token):
            continue
        token_sense_origin = token.layers.get(SENSE_ORIGIN, None)
        if not (token_sense_origin and "FIRST" in token_sense_origin):
            continue
        lemma = token.layers.get(L_LEMMA, None)
        pos = token.layers.get(L_POS, None)
        print(f"    [TOKEN] FIRST detected: {lemma} | idx: {getattr(token, 'token_index', None)}")
        context = sentence.text
        marked_sentence = build_marked_sentence(context, lemma)
        print(f"    Token lemma: {lemma}, pos: {pos}")
        # Step 1: Check new_senses_df for this lemma/pos
        candidates = new_senses_df[(new_senses_df[S_LEMMA] == lemma) & (new_senses_df[S_POS] == pos)]
        print(f"      New senses candidates: {len(candidates)}")
        senses_block = build_senses_block(candidates) if not candidates.empty else ""
        senses_candidates = ";".join(candidates[S_ID].tolist())
        senses_count = str(len(candidates))
        if not candidates.empty:
            result = selection_chain.invoke({
                "sentence": marked_sentence,
                "word": lemma,
                "senses_block": senses_block
            })
            result = json.loads(result)
            explanation = result.get("explanation", "")
            if result.get("sense_id") != "NEW_SENSE":
                match = candidates[candidates[S_ID] == result.get("sense_id")]
                if not match.empty:
                    token.layers[SENSE_ID_FIELD] = f"{match.iloc[0][S_ID]}[{n}]"
                    token.layers[SENSE_ORIGIN] = 'AI-NEW-REPO'
                    token.layers[SENSE_AINOTES_FIELD] = f"{explanation}[{n}]"
                    token.layers[SENSE_LIST_FIELD] = f"{senses_candidates}[{n}]"
                    token.layers[SENSE_COUNT_FIELD] = f"{senses_count}[{n}]"
                    n += 1
                    continue
        # Step 2: RAG over main repo
        # Always include 'embedding' column in filtered DataFrame
        filter_cols = [S_ID, S_DEFINITION, S_TYPE, S_POS, 'embedding']
        filtered = senses_df[filter_cols]
        if pos and S_POS in senses_df.columns:
            filtered = filtered[filtered[S_POS] == pos]
        print(f"      RAG candidate senses: {len(filtered)}")
        if len(filtered) == 0:
            print("      WARNING: No RAG candidates found for this token.")
        # --- Embedding lookup and update logic ---
        context_emb = id_to_emb.get(lemma) if lemma in id_to_emb else None
        if context_emb is None:
            context_emb = embed_text(context)
            # Add the new embedding to id_to_emb for future use
            id_to_emb[lemma] = context_emb
        filtered = filtered.copy()  # Avoid SettingWithCopyWarning
        filtered['sim'] = filtered['embedding'].apply(lambda x: cosine_similarity(context_emb, x))
        top_senses = filtered.sort_values('sim', ascending=False).head(TOP_N)
        # Print the top five most semantically relevant candidates after similarity calculation
        if len(top_senses) > 0:
            print("      Top RAG candidates (by similarity):")
            print(top_senses[[S_ID, S_DEFINITION, S_TYPE, S_POS, 'sim']].head())
        senses_block = build_senses_block(top_senses)
        senses_candidates = ";".join(top_senses[S_ID].tolist())
        senses_count = str(len(top_senses))
        result = selection_chain.invoke({
            "sentence": marked_sentence,
            "word": lemma,
            "senses_block": senses_block
        })
        result = json.loads(result)
        explanation = result.get("explanation", "")
        if result.get("sense_id") != "NEW_SENSE":
            match = top_senses[top_senses[S_ID] == result.get("sense_id")]
            if not match.empty:
                best = match.iloc[0]
                token.layers[SENSE_ID_FIELD] = f"{best[S_ID]}[{n}]"
                token.layers[SENSE_ORIGIN] = 'RAG-OLD-REPO'
                token.layers[SENSE_AINOTES_FIELD] = f"{explanation}[{n}]"
                token.layers[SENSE_LIST_FIELD] = f"{senses_candidates}[{n}]"
                token.layers[SENSE_COUNT_FIELD] = f"{senses_count}[{n}]"
                n += 1
                continue
        # Step 3: Generate new sense
        result = generation_chain.invoke({
            "sentence": marked_sentence,
            "word": lemma,
            "pos": pos or ""
        })
        result = json.loads(result)
        explanation = result.get("explanation", "")
        existing_ids = set(senses_df[S_ID]).union(set(new_senses_df[S_ID]))
        new_id = generate_ai_sense_id(existing_ids)
        new_entry = {
            S_ID: new_id,
            S_LEMMA: lemma,
            S_DEFINITION: result['definition'],
            S_TYPE: None,
            S_POS: pos,
            'source': 'AI-GENERATED'
        }
        new_senses_df = pd.concat([new_senses_df, pd.DataFrame([new_entry])], ignore_index=True)
        log_new_senses(pd.DataFrame([new_entry]), new_sense_log_path)
        token.layers[SENSE_ID_FIELD] = f"{new_id}[{n}]"
        token.layers[SENSE_ORIGIN] = 'AI-GENERATED'
        token.layers[SENSE_AINOTES_FIELD] = f"{explanation}[{n}]"
        token.layers[SENSE_LIST_FIELD] = f"{new_id}[{n}]"
        token.layers[SENSE_COUNT_FIELD] = "1"
        n += 1


Processing sentence 1/279: Taj bod podelili su na jednake delove vozači koji su imali isti najbrži krug.
  Number of MWEs: 0
  Number of tokens: 15
    [TOKEN] FIRST detected: bod | idx: 2
    Token lemma: bod, pos: N
      New senses candidates: 0
      RAG candidate senses: 5030
      Top RAG candidates (by similarity):
               senseID                                         definition  \
8609  ENG30-08293490-n       Grupa motornih vozila pod istim vlasništvom.   
7173  ENG30-04463983-n  Par paralelnih šina po kijima se kreće vozilo ...   
4415  ENG30-01085098-n                Rezultat parcelisanja ili deljenja.   
7606  ENG30-05781800-n           podela u međusobno isključive kategorije   
8680  ENG30-08417801-n        povorka kopnenih vozila koja putuju zajedno   

     Type pos       sim  
8609    S   N  0.485851  
7173    S   N  0.479432  
4415    S   N  0.461036  
7606    S   N  0.450132  
8680    S   N  0.439309  
      Top RAG candidates (by similarity):
              

In [11]:
from writers import IncetprionWebAnnoTSVWriter

# Save the updated new_senses_df to a file
new_senses_path = OUTPUT_DIR / "new_senses_df.xlsx"
new_senses_df.to_excel(new_senses_path, index=False)

# Save the updated senses_df to a file
senses_path = OUTPUT_DIR / "updated_senses_df.xlsx"
senses_df.to_excel(senses_path, index=False)

# Save the updated sentences with new senses
updated_sentences_path = OUTPUT_DIR / "updated_filtered_sentences_with_senses.tsv"

writer = IncetprionWebAnnoTSVWriter(filtered_sentences)

writer.save(updated_sentences_path)
print(f"Updated sentences saved to {updated_sentences_path}")



Updated sentences saved to e:\Github\LexiSense-SR\output\updated_filtered_sentences_with_senses.tsv


# Unified Second-Round Sense Assignment Pipeline: Logic, Conventions, and Traceability

This section implements a **unified, single-pass sense assignment pipeline** for both multiword expressions (MWEs) and single tokens, designed for clarity, maintainability, and robust traceability. The pipeline is fully aligned with best practices and the conventions established in the first-round pipeline. Key features and conventions:

- **Config-Driven Field Names:** All field names for sense assignment (e.g., `SENSE_ID_FIELD`, `SENSE_AINOTES_FIELD`, `SENSE_LIST_FIELD`, `SENSE_COUNT_FIELD`) are imported from `config.py` and used consistently throughout the pipeline. This ensures maintainability and reduces the risk of errors due to hardcoded strings.

- **Support for MWEs and Single Tokens:** The logic handles both MWEs and single tokens in a unified loop, ensuring that each entity (whether a group of tokens or a single token) is processed according to the same conventions.

- **Unique Indexing for Traceability:** Every sense assignment (whether from the main repo, new sense repo, or LLM generation) is tagged with a unique index `[n]` that is incremented for each entity. This index is appended to all relevant fields, enabling precise traceability of each assignment and its corresponding LLM explanation.

- **Comprehensive Metadata Storage:** For each token (or MWE token group), the following fields are stored using the config-defined names:
  - Sense ID (with unique index)
  - Sense origin (AI-NEW-REPO, RAG-OLD-REPO, or AI-GENERATED)
  - LLM explanation (with unique index)
  - Sense candidate list (with unique index)
  - Sense candidate count (with unique index)

- **Three-Step Assignment Logic:**
  1. **Check New Senses Repo:** If a matching sense exists in the in-memory new senses DataFrame, use the LLM to select among candidates. If a match is found, assign it and log the explanation.
  2. **RAG over Main Repo:** If no match in the new senses, perform a semantic similarity search (RAG) over the main sense repository. Use the LLM to select the best candidate. If a match is found, assign it and log the explanation.
  3. **LLM Generation:** If no suitable sense is found, prompt the LLM to generate a new sense definition. Assign a new unique sense ID, log the new sense, and store all metadata.

- **Robust Logging:** All new senses generated by the LLM are logged in `ai_generated_senses.jsonl` for later review and auditability.

- **Alignment with First-Round Pipeline:** The conventions for field usage, logging, and traceability are fully aligned with the first-round pipeline (`ChatGPT_sense.ipynb`), ensuring consistency across annotation rounds.

- **Detailed Explanations:** Each code cell is preceded by a markdown cell explaining its purpose, logic, and conventions, making the pipeline easy to understand and maintain for future users.

**Best Practices:**
- Always use field names from `config.py`.
- Increment the unique index `[n]` for every entity (MWE or token) processed.
- Store all LLM explanations and sense metadata with the unique index for traceability.
- Log all new senses for review.

The following code cell implements this pipeline.

# Notes on Logging and Traceability

- All new senses generated by the LLM are logged in `ai_generated_senses.jsonl` for later review and possible integration into the main sense repository.
- The pipeline ensures that only truly novel senses are added, and all actions (selection, generation, assignment) are traceable via the token layers and logs.
- Adjust the similarity threshold or top-N as needed for your use case and desired precision/recall tradeoff.

# Notes
- All new senses are logged in `ai_generated_senses.jsonl` for later review.
- The process ensures that only truly novel senses are added, and all actions are traceable.
- Adjust the similarity threshold as needed for your use case.